# Task A (Colab Edition)

This standalone notebook mirrors the Task A benchmarking pipeline without relying on repository modules. Run it in Google Colab (or any fresh runtime) by executing the cells top to bottom.

In [ ]:
%%capture
!pip install -q pandas numpy scikit-learn xgboost transformers datasets tqdm beautifulsoup4 lxml huggingface_hub joblib matplotlib requests psutil

In [ ]:
from __future__ import annotations

import json
import os
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import psutil
from bs4 import BeautifulSoup
from huggingface_hub import hf_hub_download
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
REPORT_DIR = PROJECT_ROOT / "reports"
for path in (DATA_DIR, PROCESSED_DIR, ARTIFACT_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {PROJECT_ROOT}")

In [ ]:
# -----------------------------------------------------------------------------
# Helper utilities (serialization, metrics, text cleaning)
# -----------------------------------------------------------------------------

from typing import Optional


def set_random_seed(seed: int = 42) -> None:
    import random

    random.seed(seed)
    np.random.seed(seed)


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as fh:
        json.dump(data, fh, indent=2)


def save_pickle(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(obj, path)


def compute_metrics(y_true: Iterable[int], y_pred: Iterable[int], y_prob: Optional[Iterable[float]] = None) -> Dict[str, float]:
    y_true = np.asarray(list(y_true))
    y_pred = np.asarray(list(y_pred))
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    if y_prob is not None:
        y_prob = np.asarray(list(y_prob))
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
        except ValueError:
            metrics['roc_auc'] = float('nan')
    return metrics


def strip_html(text: Optional[str]) -> str:
    if not text:
        return ''
    soup = BeautifulSoup(text, 'lxml')
    return soup.get_text(separator=' ').strip()


def normalize_text(text: Optional[str]) -> str:
    if not text:
        return ''
    import re

    text = text.lower()
    text = re.sub(r'https?://\S+', ' <URL> ', text)
    text = re.sub(r'[\w\.-]+@[\w\.-]+', ' <EMAIL> ', text)
    text = re.sub(r'\b\d+(?:\.\d+)?\b', ' <NUMBER> ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def clean_text_series(series: pd.Series) -> pd.Series:
    return series.map(lambda txt: normalize_text(strip_html(txt)))

In [ ]:
# -----------------------------------------------------------------------------
# Dataset download + loading helpers
# -----------------------------------------------------------------------------

@dataclass
class DatasetSource:
    name: str
    filename: str
    repo_id: str
    repo_filename: str
    max_rows_env: Optional[str] = None


DATA_SOURCES = (
    DatasetSource(
        name='zefang_liu',
        filename='zefang_liu.csv',
        repo_id='zefang-liu/phishing-email-dataset',
        repo_filename='Phishing_Email.csv',
    ),
    DatasetSource(
        name='cyradar',
        filename='cyradar.csv',
        repo_id='huynq3Cyradar/Phishing_Detection_Dataset',
        repo_filename='combined_reduced.csv',
        max_rows_env='CYRADAR_MAX_ROWS',
    ),
)


def _maybe_int(value: Optional[str]) -> Optional[int]:
    if not value:
        return None
    try:
        parsed = int(value)
    except ValueError:
        raise ValueError(f'Invalid integer value: {value}')
    return parsed if parsed > 0 else None


def _download_with_progress(source: DatasetSource) -> Path:
    destination = DATA_DIR / source.filename
    if destination.exists():
        return destination
    tmp_path = destination.with_suffix('.tmp')
    print(f'Downloading {source.name} via Hugging Face ...')
    local_path = hf_hub_download(
        repo_id=source.repo_id,
        filename=source.repo_filename,
        repo_type='dataset',
        resume_download=True,
    )
    shutil.copyfile(local_path, tmp_path)
    tmp_path.replace(destination)
    return destination


def download_datasets() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for source in DATA_SOURCES:
        _download_with_progress(source)


EXPECTED_COLUMNS = {
    'text': ['text', 'body', 'email', 'message', 'email text', 'content'],
    'label': ['label', 'is_phishing', 'target', 'class', 'phishing', 'email type', 'type'],
    'id': ['id', 'email_id', 'message_id', 'unnamed: 0'],
}


ROW_LIMITS = {
    'cyradar': {
        'env': 'CYRADAR_MAX_ROWS',
        'caps': ((32, 3_000_000), (24, 2_000_000), (16, 1_000_000)),
    }
}


def _resolve_column(columns: Tuple[str, ...], candidates: List[str], default: str) -> str:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise KeyError(f'Could not find column for {default}: {columns}')


def _row_cap(name: str) -> Optional[int]:
    config = ROW_LIMITS.get(name)
    if not config:
        return None
    env_val = _maybe_int(os.getenv(config['env']))
    if env_val:
        return env_val
    total_gb = psutil.virtual_memory().total / (1024 ** 3)
    for ceiling, cap in config['caps']:
        if total_gb <= ceiling:
            return cap
    return None


def _normalize_labels(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(int).apply(lambda x: 1 if int(x) == 1 else 0)
    normalized = series.astype(str).str.strip().str.lower()
    mapping = {
        '1': 1,
        '0': 0,
        'phishing': 1,
        'phishing email': 1,
        'malicious': 1,
        'spam': 1,
        'safe': 0,
        'safe email': 0,
        'legitimate': 0,
        'legitimate email': 0,
        'ham': 0,
    }
    mapped = normalized.map(mapping)
    if mapped.isna().any():
        unknown = sorted(set(normalized[mapped.isna()]))
        raise ValueError(f'Unrecognized label values (sample): {unknown[:5]}')
    return mapped.astype(int)


def load_datasets() -> pd.DataFrame:
    frames = []
    for source in DATA_SOURCES:
        path = DATA_DIR / source.filename
        if not path.exists():
            raise FileNotFoundError(f'Missing dataset: {path}')
        kwargs = {}
        cap = _row_cap(source.name)
        if cap:
            kwargs['nrows'] = cap
            print(f'Loading {source.name} with row cap {cap:,}')
        df = pd.read_csv(path, **kwargs)
        cols = tuple(df.columns.str.lower())
        df.columns = cols
        text_col = _resolve_column(cols, EXPECTED_COLUMNS['text'], 'text')
        label_col = _resolve_column(cols, EXPECTED_COLUMNS['label'], 'label')
        id_col = None
        for candidate in EXPECTED_COLUMNS['id']:
            if candidate in cols:
                id_col = candidate
                break
        standardized = pd.DataFrame({
            'text': df[text_col].astype(str),
            'label': _normalize_labels(df[label_col]),
        })
        if id_col:
            standardized.insert(0, 'id', df[id_col].astype(str))
        else:
            standardized.insert(0, 'id', standardized.index.astype(str))
        standardized.insert(1, 'source', source.name)
        frames.append(standardized)
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.dropna(subset=['text', 'label'])
    return combined

In [ ]:
# -----------------------------------------------------------------------------
# Preprocessing + splits
# -----------------------------------------------------------------------------

def preprocess_and_split(df: pd.DataFrame, test_size: float = 0.1, val_size: float = 0.1, seed: int = 42):
    set_random_seed(seed)
    df = df.copy()
    df['clean_text'] = clean_text_series(df['text'])
    df = df.drop_duplicates(subset=['clean_text']).reset_index(drop=True)
    train_val, test = train_test_split(df, test_size=test_size, stratify=df['label'], random_state=seed)
    relative_val = val_size / (1 - test_size)
    train, val = train_test_split(train_val, test_size=relative_val, stratify=train_val['label'], random_state=seed)

    for name, split in (('train', train), ('val', val), ('test', test)):
        path = PROCESSED_DIR / f'{name}.csv'
        split.to_csv(path, index=False)
        print(f'Saved {name} split -> {path} ({len(split):,} rows)')
    return train, val, test

In [ ]:
# -----------------------------------------------------------------------------
# Training + evaluation helpers
# -----------------------------------------------------------------------------

@dataclass
class ModelConfig:
    name: str
    pipeline: Pipeline
    param_grid: Dict[str, Iterable[Any]]


def build_model_configs() -> List[ModelConfig]:
    configs: List[ModelConfig] = []

    logreg = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=40000, ngram_range=(1, 2))),
        ('clf', LogisticRegression(max_iter=1500, class_weight='balanced', solver='liblinear')),
    ])
    logreg_grid = {
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'tfidf__max_df': [0.85, 0.95],
        'clf__C': [0.5, 1.0, 2.0],
        'clf__penalty': ['l1', 'l2'],
    }
    configs.append(ModelConfig('logistic_regression', logreg, logreg_grid))

    svm = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=40000, ngram_range=(1, 2))),
        ('clf', CalibratedClassifierCV(base_estimator=LinearSVC(class_weight='balanced'), cv=3, method='sigmoid')),
    ])
    svm_grid = {
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'clf__base_estimator__C': [0.25, 0.5, 1.0],
    }
    configs.append(ModelConfig('linear_svm', svm, svm_grid))

    rf = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=30000)),
        ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=42)),
    ])
    rf_grid = {
        'clf__n_estimators': [200, 400],
        'clf__max_depth': [None, 30],
        'clf__max_features': ['sqrt', 'log2'],
    }
    configs.append(ModelConfig('random_forest', rf, rf_grid))

    xgb = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50000)),
        ('clf', XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            max_depth=6,
            n_estimators=400,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            n_jobs=-1,
            reg_lambda=1.0,
        )),
    ])
    xgb_grid = {
        'clf__max_depth': [4, 6],
        'clf__learning_rate': [0.05, 0.1],
        'clf__subsample': [0.8, 0.9],
        'clf__colsample_bytree': [0.7, 0.9],
    }
    configs.append(ModelConfig('xgboost', xgb, xgb_grid))

    return configs


def _get_probabilities(pipeline: Pipeline, texts: pd.Series) -> np.ndarray:
    clf = pipeline.named_steps['clf']
    if hasattr(pipeline, 'predict_proba'):
        return pipeline.predict_proba(texts)[:, 1]
    if hasattr(clf, 'predict_proba'):
        return pipeline.predict_proba(texts)[:, 1]
    if hasattr(clf, 'decision_function'):
        decision = pipeline.decision_function(texts)
        return 1 / (1 + np.exp(-decision))
    raise AttributeError('Classifier does not expose probability estimates')


def train_models(train_df: pd.DataFrame, val_df: pd.DataFrame, seed: int = 42) -> Dict[str, Dict[str, float]]:
    set_random_seed(seed)
    X_train, y_train = train_df['clean_text'], train_df['label']
    X_val, y_val = val_df['clean_text'], val_df['label']

    ml_dir = ARTIFACT_DIR / 'ml'
    ml_dir.mkdir(parents=True, exist_ok=True)

    metrics_summary: Dict[str, Dict[str, float]] = {}
    best_score = -np.inf
    best_pipeline: Optional[Pipeline] = None
    best_name = None

    for config in build_model_configs():
        print(f'Training {config.name} ...')
        search = GridSearchCV(
            config.pipeline,
            param_grid=config.param_grid,
            scoring='f1',
            cv=3,
            n_jobs=-1,
            verbose=1,
        )
        search.fit(X_train, y_train)
        model = search.best_estimator_
        y_pred = model.predict(X_val)
        try:
            y_prob = _get_probabilities(model, X_val)
        except AttributeError:
            y_prob = None
        metrics = compute_metrics(y_val, y_pred, y_prob)
        metrics_summary[config.name] = metrics
        save_pickle(model, ml_dir / f'{config.name}.joblib')
        if metrics['f1'] > best_score:
            best_score = metrics['f1']
            best_pipeline = model
            best_name = config.name
        print(f"{config.name} F1={metrics['f1']:.4f}")

    if best_pipeline is not None:
        save_pickle(best_pipeline, ARTIFACT_DIR / 'best_model.joblib')
        save_json({'model': best_name, 'metrics': metrics_summary[best_name]}, ARTIFACT_DIR / 'best_model_meta.json')
        print(f'Best model: {best_name} (F1={best_score:.4f})')
    save_json(metrics_summary, REPORT_DIR / 'ml_metrics.json')
    return metrics_summary


def evaluate_models(test_df: pd.DataFrame) -> Dict[str, Dict[str, float]]:
    ml_dir = ARTIFACT_DIR / 'ml'
    if not ml_dir.exists():
        raise FileNotFoundError('No trained models found. Run the training cell first.')
    evaluations = {}
    for path in ml_dir.glob('*.joblib'):
        model = joblib.load(path)
        y_true = test_df['label']
        y_pred = model.predict(test_df['clean_text'])
        try:
            y_prob = _get_probabilities(model, test_df['clean_text'])
        except AttributeError:
            y_prob = None
        metrics = compute_metrics(y_true, y_pred, y_prob)
        evaluations[path.stem] = metrics
        print(f"[test] {path.stem}: {metrics}")
    save_json(evaluations, REPORT_DIR / 'test_metrics.json')
    return evaluations

In [ ]:
# -----------------------------------------------------------------------------
# Execute the pipeline
# -----------------------------------------------------------------------------

download_datasets()
full_df = load_datasets()
print(f'Combined dataset size: {len(full_df):,} rows')
train_df, val_df, test_df = preprocess_and_split(full_df)
metrics_summary = train_models(train_df, val_df)
metrics_summary

In [ ]:
test_metrics = evaluate_models(test_df)
test_metrics